#### Import/install

##### Setup

In [1]:
# ### CELL 11: Install Required Libraries (stable Windows/Anaconda setup)
# # %load_ext autoreload
# # %autoreload 2

# # Use user site-packages to avoid permission/lock issues in base Anaconda folders.
# %pip install --upgrade pip
# %pip install --user --index-url https://download.pytorch.org/whl/cpu torch torchvision torchaudio

# %pip install xlsxwriter
# %pip install optuna
# %pip install stable-baselines3 gymnasium sb3-contrib
# %pip install seaborn

# %pip install plotly
# %pip install pulp
# %pip install nbformat
# %pip install scikit-learn

# print("After package updates, restart the kernel once before imports.")

In [ ]:
### Import Libraries
import copy
import json
import os
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

import torch as T
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler


IN_COLAB = os.path.exists('/content')
IN_MAC = sys.platform == "darwin"
IN_WINDOWS = sys.platform == "win32"

# 1. NVIDIA CUDA (Linux/Windows)
if T.cuda.is_available():
    device = T.device("cuda")
    print(f"Using NVIDIA CUDA GPU: {T.cuda.get_device_name(0)}")
# 2. Apple Silicon MPS (Mac M1/M2/M3/M4)
elif T.backends.mps.is_available():
    device = T.device("mps")
    print("Using Apple Silicon MPS Acceleration")
# 3. CPU fallback
else:
    device = T.device("cpu")
    print("Using CPU")

import importlib
import Environment as environment_module
importlib.reload(environment_module)
HouseholdEnvironment = environment_module.HouseholdEnvironment

from Basic_Functions import cumulative_interval_price_series
from MILP_Benchmark import run_milp_benchmark
from Plotting_Functions import plotMultiY

from types import SimpleNamespace


##### Inport

In [4]:
### CELL 13: Dataset and column selection (single source of truth)
HOUSEHOLD_ID = 29
DATASET_NAME = "Fluvius"
SMP_COUNTRY_ID = "Slovenia"

# Use raw dataset column names directly (no aliasing in loader/environment).
PRICE_COLUMN = "SMP"
GENERATION_COLUMN = "Feed_In_Volume_kWh"
CONSUMPTION_COLUMN = "Consumption_Volume_kWh"

print(
    f"Dataset={DATASET_NAME}, household={HOUSEHOLD_ID}, SMP country={SMP_COUNTRY_ID}, "
    f"price={PRICE_COLUMN}, generation={GENERATION_COLUMN}, consumption={CONSUMPTION_COLUMN}"
)

Dataset=Fluvius, household=29, SMP country=Slovenia, price=SMP, generation=Feed_In_Volume_kWh, consumption=Consumption_Volume_kWh


In [ ]:
#### Load Dataset via Centralized Loader

import importlib
import Data_Loader as dl
dl = importlib.reload(dl)

print("Available household columns:", dl.household_column_names(HOUSEHOLD_ID, dataset=DATASET_NAME))

input_data = dl.load_household_data(
    HOUSEHOLD_ID,
    dataset=DATASET_NAME,
    print_column_names=True,
)

# Replace/fill household SMP with selected country SMP profile.
smp_country = dl.load_smp_data(SMP_COUNTRY_ID)
smp_aligned = smp_country.reindex(input_data.index, method="ffill")

# Always clean and convert SMP values before assigning to the dataset.
smp_series = pd.to_numeric(smp_aligned[PRICE_COLUMN], errors="coerce").ffill().bfill()

# Auto-detect legacy EUR/MWh inputs and convert to EUR/kWh.
# Typical EUR/kWh values are usually < 2.0, while EUR/MWh are often tens/hundreds.
unit_threshold = 2.0
smp_q95_abs = float(smp_series.abs().quantile(0.95))
smp_scale = 1000.0 if smp_q95_abs > unit_threshold else 1.0
smp_unit_in = "EUR/MWh" if smp_scale == 1000.0 else "EUR/kWh"

input_data[PRICE_COLUMN] = (smp_series / smp_scale).astype(float)

print(
    f"SMP normalization: detected {smp_unit_in} (q95={smp_q95_abs:.3f}), "
    f"applied scale 1/{int(smp_scale)} -> EUR/kWh."
)
print(
    f"SMP range after normalization [EUR/kWh]: "
    f"min={input_data[PRICE_COLUMN].min():.6f}, max={input_data[PRICE_COLUMN].max():.6f}"
)

price_series = input_data[PRICE_COLUMN]
generation_series = input_data[GENERATION_COLUMN]
consumption_series = input_data[CONSUMPTION_COLUMN]

print(
    f"Loaded household {HOUSEHOLD_ID} from {DATASET_NAME} with "
    f"{len(input_data)} intervals and SMP from {SMP_COUNTRY_ID}."
)


In [ ]:
### Split Dataset into Training and Testing Sets
test_size = 1 / 3
train_data, test_data = train_test_split(
    input_data,
    test_size=test_size,
    shuffle=False,
)


#### Basic functions

In [ ]:
### Model and training parameters

# Battery transfer limits are specified per 15-minute step.
BASE_STEP_MINUTES = 15.0
MAX_CHARGE_KWH_BASE = 1.5
MAX_DISCHARGE_KWH_BASE = 1.5

# Infer dataset resolution and adapt all per-step parameters.
step_minutes = (
    input_data.index.to_series().sort_values().diff().dropna().dt.total_seconds().median() / 60.0
)
if not np.isfinite(step_minutes) or step_minutes <= 0:
    step_minutes = BASE_STEP_MINUTES

steps_per_day = int(round((24 * 60) / step_minutes))
step_scale = float(step_minutes / BASE_STEP_MINUTES)

max_charge_kwh = MAX_CHARGE_KWH_BASE * step_scale
max_discharge_kwh = MAX_DISCHARGE_KWH_BASE * step_scale
battery_capacity_kwh = 20
charge_efficiency = 0.95
discharge_efficiency = 0.95

fc1_dims = 64 * 4
fc2_dims = 32 * 4
fc3_dims = 16 * 4

lr = 0.001
gamma = 0.99
epsilon_start = 1.0
epsilon_end = 0.01
epsilon_decay = 4e-5
batch_size = steps_per_day
replace_target = steps_per_day * 7

weight_decay = 1e-4

# Reward weights: state-of-charge band, price arbitrage, electricity cost.
reward_weight_soc = 0
reward_weight_arbitrage = 0
reward_weight_cost = 1

GENERATE_DQN_INVOICE      = False
GENERATE_BASELINE_INVOICE = False
GENERATE_MILP_INVOICE     = False

print(
    f"Detected step: {step_minutes:.1f} min | steps_per_day={steps_per_day} | "
    f"max_charge_kwh={max_charge_kwh:.3f} | max_discharge_kwh={max_discharge_kwh:.3f}"
)


In [46]:
### CELL 23: Normalize the dataset

# --- 1. CONFIGURATION ---
# Options: MinMaxScaler, StandardScaler, RobustScaler
ScalerClass = RobustScaler

# Configuration for columns
shared_cols = [GENERATION_COLUMN, CONSUMPTION_COLUMN]
independent_cols = [c for c in train_data.columns if c not in shared_cols + [PRICE_COLUMN]]

# --- 2. INITIALIZE SCALERS ---
# We use **kwargs so we can pass arguments only if the scaler supports them
scaler_args = {}

if ScalerClass == MinMaxScaler:
    scaler_args['feature_range'] = (0, 1)

# Create two separate instances of the CHOSEN class
scaler_shared = ScalerClass(**scaler_args)
scaler_indep = ScalerClass(**scaler_args)

# --- 3. FIT THE SCALERS ---

# A. Shared Scaler (Stacked fit)
# Flatten combined data so the scaler learns Global Min/Max or Global Mean/Std
combined_energy_data = train_data[shared_cols].values.flatten().reshape(-1, 1)
scaler_shared.fit(combined_energy_data)

# B. Independent Scaler (Normal fit)
if independent_cols:
    scaler_indep.fit(train_data[independent_cols])

# --- 4. SMP-SPECIFIC PRICE NORMALIZATION ---
def normalize_electricity_prices(data, fit=False, params=None):
    # Ensure data is 2D for the Scaler (N samples, 1 feature)
    if data.ndim == 1:
        data = data.reshape(-1, 1)

    # --- Step 1: Outlier Detection & Flagging ---
    # Fit thresholds only on train split, then reuse on test to avoid leakage.
    if fit:
        upper_threshold = np.percentile(data, 99.5)
        lower_threshold = np.percentile(data, 0.1)
    else:
        if params is None:
            raise ValueError("params must be provided when fit=False")
        upper_threshold = params['upper_threshold']
        lower_threshold = params['lower_threshold']

    # Create binary flag: 1 if it's an extreme event, 0 otherwise
    # We convert to float so it plays nice with neural network inputs
    outlier_flag = ((data > upper_threshold) | (data < lower_threshold)).astype(float)

    # --- Step 2: Log Transformation ---
    # np.maximum(data, 0) handles rare negative prices before log1p
    log_data = np.log1p(np.maximum(data, 0))

    # --- Step 3: Clipping (Winsorization) ---
    # We clip the log-data to the same percentiles used for the flag
    # This prevents the DQN gradients from exploding during those ~2 days a year
    log_lower = np.log1p(max(0, lower_threshold))
    log_upper = np.log1p(max(0, upper_threshold))
    clipped_data = np.clip(log_data, log_lower, log_upper)

    # --- Step 4: Robust Scaling ---
    # Fit scaler on train, reuse for test.
    if fit:
        scaler = RobustScaler()
        normalized_prices = scaler.fit_transform(clipped_data)
        fitted_params = {
            'upper_threshold': float(upper_threshold),
            'lower_threshold': float(lower_threshold),
            'log_lower': float(log_lower),
            'log_upper': float(log_upper),
            'scaler': scaler,
        }
    else:
        scaler = params['scaler']
        normalized_prices = scaler.transform(clipped_data)
        fitted_params = params

    # Return normalized price, outlier flag and fitted params for reuse.
    return normalized_prices, outlier_flag, fitted_params

# --- 5. APPLY TRANSFORMATION FUNCTION ---
def apply_scaling(df, scaler_shared, scaler_indep, shared_cols, indep_cols):
    df_scaled = df.copy()

    # Transform Shared Columns
    # We must transform one by one because scaler_shared expects a single column input shape
    for col in shared_cols:
        col_values = df[col].values.reshape(-1, 1)
        # flatten() ensures we put a 1D array back into the dataframe
        df_scaled[col] = scaler_shared.transform(col_values).flatten()

    # Transform Independent Columns
    if indep_cols:
        df_scaled[indep_cols] = scaler_indep.transform(df[indep_cols])

    return df_scaled

# --- 6. EXECUTE ---
train_data_norm = apply_scaling(train_data, scaler_shared, scaler_indep, shared_cols, independent_cols)
test_data_norm = apply_scaling(test_data, scaler_shared, scaler_indep, shared_cols, independent_cols)

train_smp_norm, train_smp_outlier, price_norm_params = normalize_electricity_prices(train_data[PRICE_COLUMN].values, fit=True)
test_smp_norm, test_smp_outlier, _ = normalize_electricity_prices(test_data[PRICE_COLUMN].values, fit=False, params=price_norm_params)

train_data_norm[PRICE_COLUMN] = train_smp_norm.flatten()
test_data_norm[PRICE_COLUMN] = test_smp_norm.flatten()
train_data_norm[f"{PRICE_COLUMN}_OutlierFlag"] = train_smp_outlier.flatten()
test_data_norm[f"{PRICE_COLUMN}_OutlierFlag"] = test_smp_outlier.flatten()

print(f"Using Scaler: {ScalerClass.__name__}")
print(train_data_norm.head())


Using Scaler: RobustScaler
                           Consumption_Volume_kWh  Feed_In_Volume_kWh  \
Timestamp_UTC                                                           
2024-01-01 00:00:00+00:00                5.864407           -0.118644   
2024-01-01 00:15:00+00:00                6.966102           -0.118644   
2024-01-01 00:30:00+00:00                2.203390           -0.118644   
2024-01-01 00:45:00+00:00                3.627119           -0.118644   
2024-01-01 01:00:00+00:00                5.254237           -0.118644   

                                SMP  SMP_OutlierFlag  
Timestamp_UTC                                         
2024-01-01 00:00:00+00:00 -1.925212              0.0  
2024-01-01 00:15:00+00:00 -1.925212              0.0  
2024-01-01 00:30:00+00:00 -1.925212              0.0  
2024-01-01 00:45:00+00:00 -1.925212              0.0  
2024-01-01 01:00:00+00:00 -1.924703              0.0  


#### Environment

In [ ]:
### Use Gymnasium Environment (HouseholdEnvironment)
# action_mode="discrete" keeps the legacy 5-action space the DQN is built on.
# Pass action_mode="continuous" to drive the battery setpoint directly, the way
# the MILP benchmark does.
def build_dqn_env(
    dataset,
    dataset_norm,
    episode_length,
    reset_mode="deterministic",
    observation_mode="sliding_window",
    action_mode="discrete",
    generate_monthly_invoice=False,
    generate_period_invoice=False,
    invoice_run_label=None,
):
    return HouseholdEnvironment(
        dataset=dataset,
        dataset_norm=dataset_norm,
        price_column=PRICE_COLUMN,
        generation_column=GENERATION_COLUMN,
        consumption_column=CONSUMPTION_COLUMN,
        observation_mode=observation_mode,
        reset_mode=reset_mode,
        action_mode=action_mode,
        episode_length=episode_length,
        steps_per_day=steps_per_day,
        battery_capacity_kwh=battery_capacity_kwh,
        charge_efficiency=charge_efficiency,
        discharge_efficiency=discharge_efficiency,
        max_charge_kwh=max_charge_kwh,
        max_discharge_kwh=max_discharge_kwh,
        reward_weight_soc=reward_weight_soc,
        reward_weight_arbitrage=reward_weight_arbitrage,
        reward_weight_cost=reward_weight_cost,
        generate_monthly_invoice=generate_monthly_invoice,
        generate_period_invoice=generate_period_invoice,
        invoice_run_label=invoice_run_label,
    )

print("Notebook DQN now uses HouseholdEnvironment directly")


In [48]:
### CELL 32: Prepare deterministic DQN evaluation environment
# %%time
eval_env = build_dqn_env(
    dataset=test_data,
    dataset_norm=test_data_norm,
    episode_length=len(test_data) - 1,
    reset_mode="deterministic",
    observation_mode="sliding_window",
)

print("Prepared deterministic DQN evaluation environment.")

Prepared deterministic DQN evaluation environment.


## DQN

### Setup DQN

In [ ]:
class DeepQNetwork(nn.Module):
    def __init__(self, lr, layer_dims, weight_decay=1e-4):
        super(DeepQNetwork, self).__init__()
        self.layer_dims = list(layer_dims)

        self.layers = nn.ModuleList([
            nn.Linear(in_dim, out_dim)
            for in_dim, out_dim in zip(self.layer_dims[:-1], self.layer_dims[1:])
        ])

        self.optimizer = optim.AdamW(self.parameters(), lr=lr, weight_decay=weight_decay)
        # SmoothL1 (Huber) is typically more stable than MSE for noisy TD targets.
        self.loss = nn.SmoothL1Loss(beta=1.0)

        self.device = T.device('cuda:0' if T.cuda.is_available() else 'cpu')

        nn.init.constant_(self.layers[-1].bias, 0.1)
        self.to(self.device)

    def forward(self, state):
        x = state
        for layer in self.layers[:-1]:
            x = F.relu(layer(x))
        return self.layers[-1](x)


class AgentDQN:
    def __init__(
        self,
        gamma,
        epsilon,
        lr,
        state_shape,
        network_dims,
        batch_size,
        replace_target=replace_target,
        max_mem_size=1_000_000,
        eps_end=0.05,
        eps_dec=4e-5,
        weight_decay=weight_decay,
        tau=0.005,
        start_learning_after=None,
    ):
        self.gamma = gamma
        self.epsilon = epsilon
        self.eps_min = eps_end
        self.eps_dec = eps_dec
        self.lr = lr
        self.batch_size = batch_size
        self.mem_size = max_mem_size
        self.mem_cntr = 0
        self.iter_cntr = 0
        self.replace_target = replace_target
        self.tau = float(tau) if tau is not None else 0.0

        self.state_shape = list(state_shape)
        self.network_dims = list(network_dims)
        self.n_actions = self.network_dims[-1]
        self.action_space = list(range(self.n_actions))

        self.Q_eval = DeepQNetwork(lr=lr, layer_dims=self.network_dims, weight_decay=weight_decay)
        self.Q_target = DeepQNetwork(lr=lr, layer_dims=self.network_dims, weight_decay=weight_decay)
        self.Q_target.load_state_dict(self.Q_eval.state_dict())
        self.Q_target.eval()

        if start_learning_after is None:
            self.start_learning_after = int(4 * self.batch_size)
        else:
            self.start_learning_after = int(max(start_learning_after, self.batch_size))

        self.state_memory = np.zeros((self.mem_size, *self.state_shape), dtype=np.float32)
        self.new_state_memory = np.zeros((self.mem_size, *self.state_shape), dtype=np.float32)
        self.action_memory = np.zeros(self.mem_size, dtype=np.int32)
        self.reward_memory = np.zeros(self.mem_size, dtype=np.float32)
        self.terminal_memory = np.zeros(self.mem_size, dtype=np.bool_)

    def store_transition(self, state, action, reward, state_, terminal):
        index = self.mem_cntr % self.mem_size
        self.state_memory[index] = state
        self.new_state_memory[index] = state_
        self.reward_memory[index] = reward
        self.action_memory[index] = action
        self.terminal_memory[index] = terminal

        self.mem_cntr += 1

    def choose_action(self, observation):
        if np.random.random() > self.epsilon:              # epsilon-greedy
            state = T.tensor(np.array([observation]), dtype=T.float32).to(self.Q_eval.device)
            actions = self.Q_eval.forward(state)           # Q-values for every action
            action = T.argmax(actions).item()              # greedy pick
        else:
            action = np.random.choice(self.action_space)

        return action

    def _soft_update_target(self):
        with T.no_grad():
            for target_param, eval_param in zip(self.Q_target.parameters(), self.Q_eval.parameters()):
                target_param.data.mul_(1.0 - self.tau).add_(self.tau * eval_param.data)

    def learn(self):
        if self.mem_cntr < self.start_learning_after:
            return

        # Occupied memory; once state_memory fills up for the first time it stays mem_size.
        max_mem = min(self.mem_cntr, self.mem_size)
        if max_mem < self.batch_size:
            return

        # batch_size distinct memory slots (replace=False -> no repeats)
        batch = np.random.choice(max_mem, self.batch_size, replace=False)
        batch_index = np.arange(self.batch_size, dtype=np.int32)

        state_batch = T.tensor(self.state_memory[batch], dtype=T.float32, device=self.Q_eval.device)
        new_state_batch = T.tensor(self.new_state_memory[batch], dtype=T.float32, device=self.Q_eval.device)
        action_batch_t = T.tensor(self.action_memory[batch], dtype=T.long, device=self.Q_eval.device)
        reward_batch = T.tensor(self.reward_memory[batch], dtype=T.float32, device=self.Q_eval.device)
        terminal_batch_t = T.tensor(self.terminal_memory[batch], dtype=T.float32, device=self.Q_eval.device)

        q_eval_all = self.Q_eval(state_batch)
        q_eval = q_eval_all[batch_index, action_batch_t]

        with T.no_grad():
            # Double DQN: the online net picks the action...
            next_actions = self.Q_eval(new_state_batch).argmax(dim=1, keepdim=True)

            # ...and the target net values it.
            q_next = self.Q_target(new_state_batch).gather(1, next_actions).squeeze(1)

            # Bellman target; terminal states contribute no bootstrap value.
            q_target = reward_batch + self.gamma * q_next * (1 - terminal_batch_t)

        loss = self.Q_eval.loss(q_eval, q_target).to(self.Q_eval.device)
        self.Q_eval.optimizer.zero_grad()   # clear old gradients, PyTorch accumulates otherwise
        loss.backward()                     # backpropagate the TD error
        T.nn.utils.clip_grad_norm_(self.Q_eval.parameters(), max_norm=1.0)  # gradient clipping for stability
        self.Q_eval.optimizer.step()        # apply the update

        self.iter_cntr += 1
        self.epsilon = self.epsilon - self.eps_dec if self.epsilon > self.eps_min else self.eps_min

        if self.tau > 0.0:
            self._soft_update_target()
        elif self.iter_cntr % self.replace_target == 0:
            self.Q_target.load_state_dict(self.Q_eval.state_dict())


In [ ]:
import datetime as dt
import os
import random


def _save_dqn_checkpoint(agent, checkpoint_file, network_dims, episode_idx=None, extra=None):
    payload = {
        "state_dict": agent.Q_eval.state_dict(),
        "network_dims": list(network_dims),
        "episode": int(episode_idx) if episode_idx is not None else None,
        "saved_at": dt.datetime.now().isoformat(),
    }
    if extra:
        payload.update(extra)

    tmp_file = checkpoint_file + ".tmp"
    T.save(payload, tmp_file)
    os.replace(tmp_file, checkpoint_file)


def set_global_seed(seed):
    if seed is None:
        return

    random.seed(seed)
    np.random.seed(seed)
    T.manual_seed(seed)

    if T.cuda.is_available():
        T.cuda.manual_seed_all(seed)
        T.backends.cudnn.deterministic = True
        T.backends.cudnn.benchmark = False


def _update_optimizer_lr(agent, new_lr):
    if new_lr is None:
        return

    for net in (getattr(agent, "Q_eval", None), getattr(agent, "Q_target", None)):
        if net is not None and hasattr(net, "optimizer"):
            for group in net.optimizer.param_groups:
                group["lr"] = float(new_lr)

    agent.lr = float(new_lr)


def train_dqn(epsilon=None, reset=False, episodes=52, epsilon_decay=None,
              epsilon_end=None, batch_size_override=None, training=True, random_policy=False,
              lr=None, agent=None, env=None,
              eval_epsilon=0.0, seed=None,
              deterministic_starts=False, start_learning_after=None,
              checkpoint_every=5, invoice_flag=None):
    """Train or evaluate the DQN agent on the discrete-action environment.

    Returns (agent, env) when training, and the per-step tracking lists when
    evaluating (training=False).
    """

    # Resolve defaults at call time so Optuna-updated globals are used.
    if epsilon is None:
        epsilon = epsilon_start
    if epsilon_decay is None:
        epsilon_decay = globals().get("epsilon_decay")
    if epsilon_end is None:
        epsilon_end = globals().get("epsilon_end")
    if batch_size_override is None:
        batch_size_override = batch_size
    if lr is None:
        lr = globals().get("lr")
    if invoice_flag is None:
        invoice_flag = globals().get("GENERATE_DQN_INVOICE", False)

    set_global_seed(seed)

    n_episodes = episodes
    save_checkpoint = True
    load_checkpoint = (agent is None)

    data = train_data if training else test_data
    data_norm = train_data_norm if training else test_data_norm

    if not training:
        save_checkpoint = False
        epsilon = 0
        epsilon_end = 0
        n_episodes = 1
        steps_per_episode = len(data) - 1
    else:
        steps_per_episode = steps_per_day * 7

    if random_policy:
        epsilon = 1
        epsilon_end = 1

    # Build gym environment if not provided.
    if env is None or reset:
        env_reset_mode = "deterministic" if (not training or deterministic_starts) else "random"
        env = build_dqn_env(
            dataset=data,
            dataset_norm=data_norm,
            episode_length=steps_per_episode,
            reset_mode=env_reset_mode,
            observation_mode="sliding_window",
            action_mode="discrete",
            # Invoicing only makes sense for a deterministic chronological
            # pass -- on for the eval env (not training), off for training.
            generate_monthly_invoice=(not training) * invoice_flag,
            generate_period_invoice=(not training) * invoice_flag,
            invoice_run_label=("dqn_eval" if (not training) and invoice_flag else None),
        )

    old_agent_epsilon = None

    state_shape = [int(env.observation_space.shape[0])]
    network_dims = [state_shape[0], fc1_dims, fc2_dims, fc3_dims, int(env.action_space.n)]

    if agent is None or reset:
        agent = AgentDQN(
            gamma=gamma,
            epsilon=epsilon,
            lr=lr,
            state_shape=state_shape,
            network_dims=network_dims,
            batch_size=batch_size_override,
            eps_end=epsilon_end,
            eps_dec=epsilon_decay,
            replace_target=replace_target,
            weight_decay=weight_decay,
            start_learning_after=start_learning_after,
        )
    else:
        # Keep continuation runs consistent when reusing an already-trained agent.
        agent.epsilon = float(epsilon)
        agent.eps_min = float(epsilon_end)
        agent.eps_dec = float(epsilon_decay)
        agent.batch_size = int(batch_size_override)
        if start_learning_after is not None:
            agent.start_learning_after = int(max(start_learning_after, agent.batch_size))
        _update_optimizer_lr(agent, lr)

    dqn_reward, dqn_cost = [0], [0]
    reward_soc_hist, reward_arbitrage_hist, reward_cost_hist = [0], [0], [0]
    dqn_soc = [battery_capacity_kwh / 2]

    checkpoint_dir = "DQN"
    if not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)
    checkpoint_file = os.path.join(checkpoint_dir, "dqn_model")

    # Reset mode must start from clean weights even if a previous checkpoint exists.
    if reset:
        load_checkpoint = False
        if os.path.exists(checkpoint_file):
            try:
                os.remove(checkpoint_file)
                print("Reset requested: removed old DQN checkpoint.")
            except OSError as exc:
                print(f"Reset requested: could not remove old checkpoint ({exc}). "
                      "Continuing without loading.")

    ckpt_dims = None
    state_dict = None
    if load_checkpoint and os.path.exists(checkpoint_file):
        ckpt = T.load(checkpoint_file, weights_only=False)

        # Backward compatibility: old checkpoints may be plain state_dict.
        if isinstance(ckpt, dict) and "state_dict" in ckpt:
            state_dict = ckpt["state_dict"]
            ckpt_dims = ckpt.get("network_dims", None)
        else:
            state_dict = ckpt
            ckpt_dims = None

    if load_checkpoint and state_dict is not None:
        if (ckpt_dims is None) or (list(ckpt_dims) == list(network_dims)):
            agent.Q_eval.load_state_dict(state_dict)
            agent.Q_target.load_state_dict(agent.Q_eval.state_dict())
            agent.Q_target.eval()
        else:
            print("Skipping checkpoint load: architecture mismatch.")
    elif load_checkpoint:
        print("Checkpoint not found; starting from randomly initialized weights.")

    if not training:
        # eval_epsilon=0.0  -> deterministic (greedy), > 0 -> stochastic robustness test
        if not random_policy:
            old_agent_epsilon = agent.epsilon
            agent.epsilon = float(eval_epsilon)
            if eval_epsilon > 0.0:
                print(f"Stochastic eval: epsilon = {eval_epsilon:.4f}")
            else:
                print("Deterministic eval: epsilon = 0.0 (greedy)")
        else:
            agent.epsilon = 1.0
            print("Random eval: epsilon = 1.0")

    print(f"Episode: {n_episodes}")
    print(f"Steps per episode: {steps_per_episode}")

    for i in range(n_episodes):
        done = False

        if not training:
            reset_options = {"reset_mode": "deterministic"}
        elif deterministic_starts:
            reset_options = {"reset_mode": "sequential", "sequential_n": max(n_episodes, 1)}
        else:
            reset_options = {"reset_mode": "random"}

        try:
            obs, info = env.reset(options=reset_options)
            print(f"Epsilon {agent.epsilon:.4f}")

            step_count = 0
            while step_count < steps_per_episode and not done:
                action = agent.choose_action(obs)
                next_obs, reward, terminated, truncated, step_info = env.step(action)
                done_next = bool(terminated or truncated)

                if training:
                    agent.store_transition(obs, action, reward, next_obs, done_next)
                    agent.learn()

                obs = next_obs
                done = done_next
                step_count += 1

                dqn_reward.append(dqn_reward[-1] + reward)
                dqn_cost.append(float(step_info.get("cumulative_payment", dqn_cost[-1])))
                dqn_soc.append(float(step_info.get("battery", dqn_soc[-1])))

                reward_components = step_info.get("reward_components", {})
                reward_soc_hist.append(float(reward_components.get("r_soc", 0.0)))
                reward_arbitrage_hist.append(float(reward_components.get("r_arbitrage", 0.0)))
                reward_cost_hist.append(float(reward_components.get("r_cost", 0.0)))

            if save_checkpoint and (((i + 1) % max(int(checkpoint_every), 1)) == 0 or (i + 1) == n_episodes):
                _save_dqn_checkpoint(
                    agent, checkpoint_file, network_dims, episode_idx=i + 1,
                    extra={"epsilon": float(agent.epsilon), "learning": bool(training)},
                )

        except Exception:
            if save_checkpoint:
                _save_dqn_checkpoint(
                    agent, checkpoint_file, network_dims, episode_idx=i,
                    extra={"epsilon": float(agent.epsilon), "learning": bool(training)},
                )
            raise

    if save_checkpoint:
        _save_dqn_checkpoint(
            agent, checkpoint_file, network_dims, episode_idx=n_episodes,
            extra={"epsilon": float(agent.epsilon), "learning": bool(training)},
        )

    if old_agent_epsilon is not None:
        agent.epsilon = old_agent_epsilon

    if not training:
        print(f"Final price: {dqn_cost[-1]:.2f} EUR")
        return dqn_reward, dqn_cost, dqn_soc, reward_soc_hist, reward_arbitrage_hist, reward_cost_hist
    return agent, env


In [ ]:
def build_plot_timeline(dataset, cumulative_series):
    return dataset.index[: max(len(cumulative_series) - 1, 0)]


def rollout_env_trace(env, policy_mode="idle", reset_options=None):
    """Roll a fixed (non-learning) policy through an environment and collect a trace.

    policy_mode:
      "idle"        -- battery unused every step (works in both action modes)
      "random"      -- uniform random action
      "alternating" -- cycle through the discrete action set (discrete mode only)
    """
    if policy_mode not in {"idle", "random", "alternating"}:
        raise ValueError(f"Unsupported policy_mode: {policy_mode}")
    if policy_mode == "alternating" and env.action_mode != "discrete":
        raise ValueError("policy_mode='alternating' requires action_mode='discrete'")

    if reset_options is None:
        reset_options = {"reset_mode": "deterministic"}

    discrete = env.action_mode == "discrete"
    idle_action = min(4, int(env.action_space.n) - 1) if discrete else np.zeros(1, dtype=np.float32)

    trace = SimpleNamespace(
        date=[],
        cost=[0.0],
        reward=[],
        soc=[float(getattr(env, "battery", 0.0))],
        interval_cost=[],
        cumulative_reward=[0.0],
        cumulative_reward_soc=[0.0],
        cumulative_reward_arbitrage=[0.0],
        cumulative_reward_cost=[0.0],
    )

    dataset_index = getattr(getattr(env, "dataset", None), "index", None)

    env.reset(options=reset_options)
    done = False
    step_counter = 0

    while not done:
        if policy_mode == "random":
            action = env.action_space.sample()
        elif policy_mode == "alternating":
            action = int(step_counter % env.action_space.n)
        else:
            action = idle_action

        _, reward, terminated, truncated, step_info = env.step(action)
        reward_components = step_info.get("reward_components", {})

        trace.cost.append(float(step_info.get("cumulative_payment", trace.cost[-1])))
        if dataset_index is not None and len(dataset_index) > 0:
            trace.date.append(dataset_index[min(step_counter, len(dataset_index) - 1)])
        else:
            trace.date.append(step_counter)
        trace.reward.append(float(reward))
        trace.soc.append(float(step_info.get("battery", trace.soc[-1])))
        trace.interval_cost.append(
            float(reward_components.get("fixed_monthly_charge_eur", 0.0))
            + float(reward_components.get("variable_cost_eur", 0.0))
        )
        trace.cumulative_reward.append(trace.cumulative_reward[-1] + float(reward))
        trace.cumulative_reward_soc.append(
            trace.cumulative_reward_soc[-1] + float(reward_components.get("r_soc", 0.0))
        )
        trace.cumulative_reward_arbitrage.append(
            trace.cumulative_reward_arbitrage[-1] + float(reward_components.get("r_arbitrage", 0.0))
        )
        trace.cumulative_reward_cost.append(
            trace.cumulative_reward_cost[-1] + float(reward_components.get("r_cost", 0.0))
        )

        done = bool(terminated or truncated)
        step_counter += 1

    return trace


### Optimization

In [ ]:
# Optimization horizons
OPT_TRAIN_EPISODES = 52           # default matches train_dqn episodes
OPT_HORIZON = steps_per_day * 7   # matches train_dqn steps_per_episode


def objective(trial, train_episodes=None):
    if train_episodes is None:
        train_episodes = OPT_TRAIN_EPISODES

    # --- Hyperparameters ---
    gamma_t = trial.suggest_float("gamma", 0.8, 0.99999, log=True)
    lr_t = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    epsilon_start_t = trial.suggest_float("epsilon_start", 0.8, 1.0)
    epsilon_end_t = trial.suggest_float("epsilon_end", 0.01, 0.15)
    total_steps = train_episodes * OPT_HORIZON
    exploration_fraction = trial.suggest_float("exploration_fraction", 0.3, 0.8)
    steps_to_decay = total_steps * exploration_fraction
    epsilon_decay_t = (epsilon_start_t - epsilon_end_t) / steps_to_decay
    batch_size_t = trial.suggest_categorical("batch_size", [32, 64, 96, 128])
    fc1_t, fc2_t, fc3_t = 512, 256, 128
    replace_target_t = trial.suggest_int("replace_target", steps_per_day * 3, steps_per_day * 30)
    weight_decay_t = trial.suggest_float("weight_decay", 0.0, 1e-3)

    # --- Split train_data into opt_train (first half) and opt_val (second half) ---
    split_idx = len(train_data) // 2
    opt_train_data = train_data.iloc[:split_idx]
    opt_train_data_norm = train_data_norm.iloc[:split_idx]
    opt_val_data = train_data.iloc[split_idx:]
    opt_val_data_norm = train_data_norm.iloc[split_idx:]

    # --- Phase 1: Train on opt_train with random reset ---
    train_env = build_dqn_env(
        dataset=opt_train_data,
        dataset_norm=opt_train_data_norm,
        episode_length=OPT_HORIZON,
        reset_mode="random",
        observation_mode="sliding_window",
        action_mode="discrete",
    )

    state_shape = [int(train_env.observation_space.shape[0])]
    network_dims = [state_shape[0], fc1_t, fc2_t, fc3_t, int(train_env.action_space.n)]

    agent = AgentDQN(
        gamma=gamma_t,
        epsilon=epsilon_start_t,
        lr=lr_t,
        state_shape=state_shape,
        network_dims=network_dims,
        batch_size=batch_size_t,
        eps_end=epsilon_end_t,
        eps_dec=epsilon_decay_t,
        weight_decay=weight_decay_t,
        replace_target=replace_target_t,
    )

    episode_rewards = []
    for episode in range(train_episodes):
        obs, _ = train_env.reset(options={"reset_mode": "random"})
        episode_reward = 0.0

        for _ in range(OPT_HORIZON):
            a = agent.choose_action(obs)
            next_obs, r, terminated, truncated, _ = train_env.step(a)
            done = bool(terminated or truncated)

            agent.store_transition(obs, a, r, next_obs, done)
            agent.learn()

            obs = next_obs
            episode_reward += r

            if done:
                break

        episode_rewards.append(episode_reward)

        # Report rolling average for pruning
        trial.report(sum(episode_rewards) / len(episode_rewards), episode)

        if trial.should_prune():
            raise optuna.TrialPruned()

    # --- Phase 2: Validate on opt_val with sequential reset (greedy) ---
    val_env = build_dqn_env(
        dataset=opt_val_data,
        dataset_norm=opt_val_data_norm,
        episode_length=len(opt_val_data) - 1,
        reset_mode="sequential",
        observation_mode="sliding_window",
        action_mode="discrete",
    )

    old_epsilon = agent.epsilon
    agent.epsilon = 0.0

    obs, _ = val_env.reset(options={"reset_mode": "sequential"})
    val_reward = 0.0

    for _ in range(len(opt_val_data) - 1):
        a = agent.choose_action(obs)
        obs, r, terminated, truncated, _ = val_env.step(a)
        val_reward += r
        if terminated or truncated:
            break

    agent.epsilon = old_epsilon

    return val_reward


study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner(n_startup_trials=20, n_warmup_steps=40),
)

print("Starting hyperparameter optimization...")
study.optimize(objective, n_trials=100, timeout=3600)

print("Optimization completed.")
print(f"Finished trials: {len(study.trials)}")
print(f"Best trial: {study.best_trial.number}")
print(f"Best value: {study.best_trial.value:.4f}")

best_params = study.best_trial.params

plot_optimization_history(study).show()
plot_param_importances(study).show()


In [ ]:
# apply best Optuna parameters
gamma = best_params['gamma']
lr = best_params['lr']
epsilon_start = best_params['epsilon_start']
epsilon_end = best_params['epsilon_end']
#epsilon_decay = best_params['epsilon_decay']
exploration_fraction = best_params['exploration_fraction']
total_steps = OPT_TRAIN_EPISODES * OPT_HORIZON
steps_to_decay = total_steps * exploration_fraction
epsilon_decay = (epsilon_start - epsilon_end) / steps_to_decay
batch_size = best_params['batch_size']
# fc1_dims = best_params['fc1_dims']
# fc2_dims = best_params['fc2_dims']
# fc3_dims = best_params['fc3_dims']
replace_target = best_params['replace_target']
weight_decay = best_params['weight_decay']

### Best param save/load

In [ ]:
if IN_COLAB:
    base_dir = Path("/content/drive/MyDrive/Colab Notebooks")
else:
    # Path(".") points to your current working directory on both Mac and Windows
    base_dir = Path(".") 

# Path combining handles the slashes automatically (e.g., '\' for Windows, '/' for Mac/Colab)
best_param_loc = base_dir / "Optimized hyperparameters" / "ausgrid123-mali-SUPER.json"

# Save best Optuna parameters to file
with open(best_param_loc, "w") as f:
    json.dump(best_params, f, indent=4)

print(f"Best parameters saved to {best_param_loc}")
print(best_params)


In [ ]:
if IN_COLAB:
    best_param_loc = "/content/Household-RL/Optimized hyperparameters/N011Fc66Bu95Bk20PoskusZVecjimNN-AUSGRID.json"
else:
    best_param_loc = "Optimized hyperparameters/N001Fc100Bu95Bk20velikNN-ausgrid123.json"
# Load best parameters from file
with open(best_param_loc, "r") as f:
    best_params = json.load(f)

gamma = best_params['gamma']
lr = best_params['lr']
epsilon_start = best_params['epsilon_start']
epsilon_end = best_params['epsilon_end']
exploration_fraction = best_params['exploration_fraction']
total_steps = OPT_TRAIN_EPISODES * OPT_HORIZON
steps_to_decay = total_steps * exploration_fraction
epsilon_decay = (epsilon_start - epsilon_end) / steps_to_decay
batch_size = best_params['batch_size']
fc1_dims = 512
fc2_dims = 256
fc3_dims = 128
replace_target = best_params['replace_target']
weight_decay = best_params['weight_decay']

print(f"Loaded best parameters from {best_param_loc}")
print(best_params)


### Using the DQN

In [ ]:
reward_weight_soc = 0
reward_weight_arbitrage = 0
reward_weight_cost = 1
charge_efficiency = 0.95
discharge_efficiency = 0.95
battery_capacity_kwh = 20


In [ ]:
#%%time
train = train_dqn(reset=True)  # , episodes=100
dqn_reward, dqn_cost, dqn_soc, reward_soc_hist, reward_arbitrage_hist, reward_cost_hist = train_dqn(training=False)
print(f'Nagrada po optimizaciji {dqn_reward[-1]}')
print(f'Cena po optimizaciji {dqn_cost[-1]}')
print(f'reward_weight_soc: {reward_weight_soc}')
print(f'reward_weight_arbitrage: {reward_weight_arbitrage}')
print(f'reward_weight_cost: {reward_weight_cost}')


In [54]:
cuda_available = T.cuda.is_available()
mps_available = T.backends.mps.is_available()

print(f"CUDA (NVIDIA) available: {cuda_available}")
print(f"MPS (Apple Silicon) available: {mps_available}")
print("-" * 30)

# Determine the best available device
if cuda_available:
    device = T.device("cuda")
    print(f"Using device: {T.cuda.get_device_name(0)}")
elif mps_available:
    device = T.device("mps")
    print("Using device: Apple Silicon GPU (MPS)")
else:
    device = T.device("cpu")
    print("Running on CPU")

# print('Current NN architecture:', (state_shape[0], fc1_dims, fc2_dims, fc3_dims, env.action_space.n))

CUDA (NVIDIA) available: False
MPS (Apple Silicon) available: False
------------------------------
Running on CPU


### Effect of training repetitions on performance (cost)

In [ ]:
# Convergence test with deterministic multi-start evaluation
reward_history = []
cost_history = []
reward_std_history = []
cost_std_history = []

# 1) Reproducibility for this experiment
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
T.manual_seed(SEED)
if T.cuda.is_available():
    T.cuda.manual_seed_all(SEED)


def evaluate_policy(agent, eval_env, n_starts=8, horizon=None):
    """Evaluate current policy deterministically from multiple fixed start points."""
    if horizon is None:
        horizon = steps_per_day * 28

    old_eps = agent.epsilon
    agent.epsilon = 0.0

    rewards = []
    costs = []

    try:
        for _ in range(n_starts):
            obs, _ = eval_env.reset(options={"reset_mode": "sequential", "sequential_n": n_starts})
            cumulative_reward = 0.0
            last_info = {}

            for _ in range(horizon):
                a = agent.choose_action(obs)
                obs, r, terminated, truncated, step_info = eval_env.step(a)
                cumulative_reward += r
                last_info = step_info
                if terminated or truncated:
                    break

            rewards.append(cumulative_reward)
            costs.append(float(last_info.get("cumulative_payment", 0.0)))
    finally:
        agent.epsilon = old_eps

    return {
        "reward_mean": float(np.mean(rewards)),
        "reward_std": float(np.std(rewards)),
        "price_mean": float(np.mean(costs)),
        "price_std": float(np.std(costs)),
    }

try:
    train_agent
    train_env
except NameError:
    train_agent, train_env = train_dqn(reset=True, episodes=10)

try:
    eval_env
except NameError:
    eval_env = build_dqn_env(
        dataset=test_data,
        dataset_norm=test_data_norm,
        episode_length=steps_per_day * 7,
        reset_mode="deterministic",
        observation_mode="sliding_window",
    )

best_score = float('inf')  # lower mean price is better
best_weights = copy.deepcopy(train_agent.Q_eval.state_dict())

train_agent, train_env = train_dqn(reset=True, episodes=10)

for i in range(40):
    # 2) Continue training for a short chunk
    train_agent, train_env = train_dqn(episodes=10, agent=train_agent, env=train_env)

    # 3) Evaluate deterministically on multiple fixed starts
    metrics = evaluate_policy(train_agent, eval_env, n_starts=8, horizon=steps_per_day * 7)

    reward_history.append(metrics['reward_mean'])
    cost_history.append(metrics['price_mean'])
    reward_std_history.append(metrics['reward_std'])
    cost_std_history.append(metrics['price_std'])

    if metrics['price_mean'] < best_score:
        best_score = metrics['price_mean']
        best_weights = copy.deepcopy(train_agent.Q_eval.state_dict())

    print(
        f"Iter {i:02d} | "
        f"Reward(mean±std): {metrics['reward_mean']:.2f} ± {metrics['reward_std']:.2f} | "
        f"Price(mean±std): {metrics['price_mean']:.2f} ± {metrics['price_std']:.2f}"
    )

# Optional: restore best model found during the trajectory
train_agent.Q_eval.load_state_dict(best_weights)
train_agent.Q_target.load_state_dict(best_weights)
train_agent.Q_target.eval()

# 4) Smoothed trend (for cleaner convergence visualization)
window = 5
reward_trend = pd.Series(reward_history).rolling(window=window, min_periods=1).mean().tolist()
cost_trend = pd.Series(cost_history).rolling(window=window, min_periods=1).mean().tolist()

print(f"\nBest mean price over checkpoints: {best_score:.2f}")
print(f"Last raw mean reward: {reward_history[-1]:.2f}")
print(f"Last smoothed mean reward (window={window}): {reward_trend[-1]:.2f}")


In [ ]:
train_agent.epsilon = 0.0  # Set to greedy for final evaluation
dqn_reward, dqn_cost, dqn_soc, reward_soc_hist, reward_arbitrage_hist, reward_cost_hist = train_dqn(
    training=False, agent=train_agent
)


In [ ]:
Y = [reward_history, reward_trend]
X = range(len(reward_history))
X_label = 'Ponovitev'
Y_label = 'Skupna Nagrada'
legend = ['Optimizirano DQN (raw)', 'Optimizirano DQN (trend)']
title = 'Optimizacija preko 10 ponovitev DQN 2L 64 32N PC nagrada'
plotMultiY(X, Y, X_label, [Y_label], legend, title, save_pdf=False)
print(f'Zadnja raw nagrada: {reward_history[-1]:.2f}')
print(f'Zadnja trend nagrada: {reward_trend[-1]:.2f}')


In [ ]:
Y = [cost_history, cost_trend]
X = range(len(cost_history))
X_label = 'Ponovitev'
Y_label = 'Skupno placilo'
legend = ['Optimizirano DQN (raw)', 'Optimizirano DQN (trend)']
title = 'Optimizacija preko 10 ponovitev DQN 2L 64 32N PC placilo'
plotMultiY(X, Y, X_label, [Y_label], legend, title, save_pdf=False)
print(f'Zadnje raw placilo: {cost_history[-1]:.2f}')
print(f'Zadnje trend placilo: {cost_trend[-1]:.2f}')


In [ ]:
print(f'Nagrada po optimizaciji {dqn_reward[-1]}')
print(f'Cena po optimizaciji {dqn_cost[-1]}')


### DQN plots

#### DQN plot setup

##### MILP
Bencmark using perfect predictions.

The teoretical optimum that can be reached by other algorithms


In [ ]:
env = build_dqn_env(
    dataset=test_data,
    dataset_norm=test_data_norm,
    episode_length=len(test_data) - 1,
    reset_mode="deterministic",
    observation_mode="sliding_window",
    action_mode="discrete",
)
# The MILP benchmark lives in MILP_Benchmark.run_milp_benchmark (single shared
# implementation used by all notebooks). It is imported in the import cell
# above; call it as:
#     df_milp = run_milp_benchmark(env, use_discrete_actions=True)
# use_discrete_actions=False solves the continuous formulation instead -- the
# same battery setpoint the environment now takes natively.
# Invoicing for the solved MILP trajectory is available there too:
#     df_milp = run_milp_benchmark(env, use_discrete_actions=True,
#                                  generate_invoice=True)


##### No battery, no panels, policy baselines
The rest of setup


In [ ]:
comparison_alternating_env = build_dqn_env(
    dataset=test_data,
    dataset_norm=test_data_norm,
    episode_length=len(test_data) - 1,
    reset_mode="deterministic",
    observation_mode="sliding_window",
    action_mode="discrete",
)

comparison_dates = build_plot_timeline(test_data, dqn_cost)


In [ ]:
alternating_trace = rollout_env_trace(comparison_alternating_env, policy_mode="alternating")

random_reward, random_cost, random_soc, random_reward_soc, random_reward_arbitrage, random_reward_cost = train_dqn(
    training=False,
    random_policy=True,
    env=build_dqn_env(
        dataset=test_data,
        dataset_norm=test_data_norm,
        episode_length=len(test_data) - 1,
        reset_mode="deterministic",
        observation_mode="sliding_window",
        action_mode="discrete",
    ),
)

print(f'Cena DQN z epsilon=1 {random_cost[-1]}')
print(f'Cena izmeničnih akcij {alternating_trace.cost[-1]}')


In [ ]:
price = test_data[PRICE_COLUMN]
generation = test_data[GENERATION_COLUMN]
consumption = test_data[CONSUMPTION_COLUMN]
zero_generation = pd.Series(0.0, index=test_data.index)
cost_no_battery = cumulative_interval_price_series(consumption, generation, env, test_data)
cost_no_pv = cumulative_interval_price_series(consumption, zero_generation, env, test_data)

Y = [cost_no_battery, cost_no_pv]
X = comparison_dates
X_label = 'Čas [leto-mesec]'
Y_label = 'Cena [Eur]'
legend = ['Brez Baterije', 'Brez Baterije in panelov']
title = 'Vpliv panelov na ceno elektrike pri odkupu po polovični ceni'
plotMultiY(X, Y, X_label, [Y_label], legend, title, save_pdf=False)

print(f"Placilo brez panelov {cost_no_pv[-1]}")
print(f"Placilo brez baterije {cost_no_battery[-1]}")


In [ ]:
no_battery_env = HouseholdEnvironment(
    dataset=test_data,
    dataset_norm=test_data_norm,
    price_column=PRICE_COLUMN,
    generation_column=GENERATION_COLUMN,
    consumption_column=CONSUMPTION_COLUMN,
    observation_mode="sliding_window",
    reset_mode="deterministic",
    action_mode="discrete",
    episode_length=len(test_data) - 1,
    steps_per_day=steps_per_day,
    battery_capacity_kwh=0,
    charge_efficiency=charge_efficiency,
    discharge_efficiency=discharge_efficiency,
    max_charge_kwh=max_charge_kwh,
    max_discharge_kwh=max_discharge_kwh,
    reward_weight_soc=reward_weight_soc,
    reward_weight_arbitrage=reward_weight_arbitrage,
    reward_weight_cost=reward_weight_cost,
    generate_monthly_invoice=GENERATE_BASELINE_INVOICE,
    generate_period_invoice=GENERATE_BASELINE_INVOICE,
    invoice_run_label="baseline_no_battery",
)

no_battery_trace = rollout_env_trace(no_battery_env, policy_mode="idle")


#### Grafi DQN analiza poskusa

In [ ]:
save_pdf = False
save_png = False
print(f'charge_efficiency: {charge_efficiency}')
print(f'discharge_efficiency: {discharge_efficiency}')
print(f'battery_capacity_kwh: {battery_capacity_kwh}')
print(f'reward_weight_soc: {reward_weight_soc}')
print(f'reward_weight_arbitrage: {reward_weight_arbitrage}')
print(f'reward_weight_cost: {reward_weight_cost}')
print(f'Nagrada po optimizaciji {dqn_reward[-1]}')
print(f'Cena po optimizaciji DQN {dqn_cost[-1]}')
print(f'gamma: {gamma} ')
print(f'lr: {lr} ')
print(f'epsilon_start: {epsilon_start} ')
print(f'epsilon_end: {epsilon_end} ')
print(f'epsilon_decay: {epsilon_decay} ')
print(f'batch_size: {batch_size} ')
print(f'fc1_dims: {fc1_dims} ')
print(f'fc2_dims: {fc2_dims} ')
print(f'fc3_dims: {fc3_dims} ')
print(f'replace_target: {replace_target} ')
print(f'weight_decay: {weight_decay} ')

# --- STEP 1: Run the MILP benchmark on your environment ---
# (run_milp_benchmark is the shared implementation in MILP_Benchmark.py)
df_milp = run_milp_benchmark(
    env, use_discrete_actions=True, generate_invoice=GENERATE_MILP_INVOICE,
)  # True -> legacy discrete actions, False -> continuous battery setpoint


In [ ]:
# --- STEP 2: Update the Plotting Variables ---
# Append the cumulative cost array from the MILP DataFrame to your list of Y values.
# We slice it using [:-1] to match the length of your other tracking variables.
Y = [
    random_cost[:-1],
    dqn_cost[:-1],
    no_battery_trace.cost[:-1],
    df_milp["Cum_Cost"].to_numpy()[:-1].tolist(),
]

X = comparison_dates
X_label = 'Čas [leto-mesec]'
Y_label = 'Cena [Eur]'

legend = ['Naključno DQN', 'Optimizirano (DQN)', 'Brez Baterije', 'MILP (Global Optimum)']

title = 'Cena elektrike v drugi polovici podatkov DQN vs MILP'

# --- STEP 3: Plot the multi-line chart ---
plotMultiY(X, Y, X_label, [Y_label], legend, title, save_pdf=save_pdf, save=save_png)

# --- STEP 4: Print the final financial comparison metrics ---
milp_cost = df_milp["Cum_Cost"].iloc[-1]
baseline_cost = no_battery_trace.cost[-1]
print(f'Placilo po optimizaciji (DQN): {dqn_cost[-1]}')
print(f'Izboljšanje z DQN v primerjavi z DQN brez baterije: '
      f'{((baseline_cost - dqn_cost[-1]) / baseline_cost) * 100:.2f}%')
print(f'Placilo brez baterije: {baseline_cost}')
print(f'Placilo z MILP (Teoretični Optimum): {milp_cost}')
print(f'Izboljšanje z MILP v primerjavi z DQN brez baterije: '
      f'{((baseline_cost - milp_cost) / baseline_cost) * 100:.2f}%')
print(f'Zaostajanje DQN v primerjavi z MILP: '
      f'{((dqn_cost[-1] - milp_cost) / milp_cost) * 100:.2f}%')
print(f'Delež teoretične izboljšave ki jo je dosegel DQN: '
      f'{((baseline_cost - dqn_cost[-1]) / (baseline_cost - milp_cost)) * 100:.2f}%')


In [ ]:
### Primerjava skupne nagrade po različnih metodah

Y = [
    no_battery_trace.cumulative_reward[:-1],
    dqn_reward[:-1],
    df_milp["Cum_RL_Reward"].to_numpy()[:-1],
]
X = comparison_dates
X_label = 'Čas [leto-mesec]'
Y_label = 'Nagrada'
legend = ['Brez baterije', 'DQN', 'MILP (global optimum)']
title = 'Primerjava nagrad baseline in DQN'
plotMultiY(X, Y, X_label, [Y_label], legend, title, save_pdf=save_pdf, save=save_png)

baseline_reward = no_battery_trace.cumulative_reward[-1]
milp_reward = df_milp["Cum_RL_Reward"].iloc[-1]
print(f'Nagrada po optimizaciji (DQN): {dqn_reward[-1]}')
print(f'Nagrada brez baterije: {baseline_reward}')
print(f'Nagrada z MILP (Teoretični Optimum): {milp_reward}')
print(f'Izboljšanje z DQN v primerjavi z brez baterije: '
      f'{((dqn_reward[-1] - baseline_reward) / abs(baseline_reward)) * 100:.2f}%')
print(f'Izboljšanje z MILP v primerjavi z brez baterije: '
      f'{((milp_reward - baseline_reward) / abs(baseline_reward)) * 100:.2f}%')
print(f'Delež teoretične izboljšave ki jo je dosegel DQN: '
      f'{((dqn_reward[-1] - baseline_reward) / (milp_reward - baseline_reward)) * 100:.2f}%')


In [ ]:
### Primerjava stanja baterije
Y = [
    random_soc[:-1],
    dqn_soc[:-1],
    alternating_trace.soc[:-1],
    df_milp["SOC_kWh"].to_numpy()[:-1],
]
X = comparison_dates
X_label = 'Čas [leto-mesec]'
Y_label = 'Napolnjenost baterije [kWh]'
legend = ['Naključno', 'Optimizirano', 'Zaporedno', 'MILP (global optimum)']
title = 'Napolnjenost baterije sistem Q spodbujevalnega učenja DQN'
plotMultiY(X, Y, X_label, [Y_label], legend, title, save_pdf=save_pdf, save=save_png)


In [ ]:
### Ogled stanja baterije
Y = [dqn_soc[:-1], df_milp["SOC_kWh"].to_numpy()[:-1]]
X = comparison_dates
X_label = 'Čas [leto-mesec]'
Y_label = 'Stanje baterije'
legend = ['Optimizirano', 'MilP']
title = 'Proporcionalna napolnjenost baterije optimizirano DQN'
plotMultiY(X, Y, X_label, [Y_label], legend, title, save_pdf=save_pdf, save=save_png)
